**🔥 Z-Ordering (Optional Boost)**
- You can use Z-ORDER BY with OPTIMIZE to cluster data by a column. This helps speed up queries that filter by that column.
- This organizes the data so that rows with similar customer_id values are stored close together—making filtering faster.
- OPTIMIZE sales_data ZORDER BY (customer_id);


### create table

In [0]:

from delta.tables import *
DeltaTable.createOrReplace(spark) \
    .tableName("tri.tri.scd2Demo4") \
    .addColumn("pk1", "INT") \
    .addColumn("pk2", "STRING") \
    .addColumn("dim1", "INT") \
    .addColumn("dim2", "INT") \
    .addColumn("dim3", "INT") \
    .addColumn("dim4", "INT") \
    .addColumn("active_status", "STRING") \
    .addColumn("start_date", "TIMESTAMP") \
    .addColumn("end_date", "TIMESTAMP") \
    .execute()

In [0]:
%sql
insert into tri.tri.scd2Demo4 values (111,'Unit1',200,500,800,400,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo4 values (222,'Unit2',900,NULL,700,100,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo4 values (333,'Unit3',300,900,250,650,'Y',current_timestamp(),'9999-12-31');

In [0]:
%sql
insert into tri.tri.scd2Demo4 values (666,'Unit6',200,500,800,400,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo4 values (777,'Unit7',900,NULL,700,100,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo4 values (888,'Unit8',300,900,250,650,'Y',current_timestamp(),'9999-12-31');

### Here we store the table into a particular location so we can perform operations on top of the path or location

In [0]:

spark.table("tri.tri.scd2Demo4") \
    .write \
    .format("delta") \
    .mode("append") \
    .save("/Volumes/tri/tri/tri_files/scd2Demo4")



In [0]:
%sql
SELECT * FROM delta.`/Volumes/tri/tri/tri_files/scd2Demo4`

### create table instance

In [0]:
from delta.tables import DeltaTable
target_table=DeltaTable.forPath(spark, "/Volumes/tri/tri/tri_files/scd2Demo4")
display(target_table.toDF())


### we can check number of files active before z order

In [0]:
display(target_table.detail())

### 1. Check Query Performance Before Z-Order

In [0]:
%sql
SELECT * from delta.`/Volumes/tri/tri/tri_files/scd2Demo4`
WHERE pk1 = 111

In [0]:
display(target_table.toDF().where("pk1=111"))


###  Apply Z-Order

In [0]:
target_table.optimize().executeZOrderBy("PK1")

In [0]:
display(target_table.optimize().executeZOrderBy("PK1"))

### we can check number of files active after z order

In [0]:
display(target_table.detail())

### 2. Check Query Performance after Z-Order

In [0]:
display(target_table.toDF().where("pk1=111"))

In [0]:
display(target_table.history())